# Silver Layer — Cleaning & Transformation

In [0]:
# Using Bronze layer tables as input for Silver processing

customers_df = spark.table("bronze_customers")
orders_df = spark.table("bronze_orders")
order_items_df = spark.table("bronze_order_item")
sellers_df = spark.table("bronze_sellers")
reviews_df = spark.table("bronze_reviews")
products_df = spark.table("bronze_products")
payments_df = spark.table("bronze_payments")
geolocation_df = spark.table("bronze_geolocation")

In [0]:
# Identifying missing values
from pyspark.sql.functions import col,when,count

def missing_values(df,df_name):
    print(f"Missing Values in {df_name}")
    df.select([count(when(col(c).isNull(),1)).alias(c) for c in df.columns]).show()

In [0]:
missing_values(customers_df,"customers")
missing_values(orders_df,"orders")
missing_values(order_items_df,"items")
missing_values(sellers_df,"seller")
missing_values(products_df,"products")
missing_values(payments_df,"payments")
missing_values(reviews_df,"reviews")
missing_values(geolocation_df,"geolocation")

- Handle Missing Values

In [0]:
# 1. Drop rows with missing values in critical columns
# These columns are essential identifiers and cannot be null
orders_df_cleaned  = orders_df.na.drop(
    subset=["customer_id","order_id","order_status"]
    )

print("Before:", orders_df.count())
print("After:", orders_df_cleaned.count())

- Fill Missing Value

In [0]:
from pyspark.sql.functions import col, when, to_timestamp, lit

orders_df_cleaned = orders_df.withColumn(
    "order_delivered_customer_date",
    when(
        col("order_delivered_customer_date").isNull(),
        to_timestamp(lit("9999-12-31"))
    ).otherwise(col("order_delivered_customer_date"))
)

# Add a flag column

orders_df_cleaned = orders_df.withColumn(
    "is_delivered",
    col("order_delivered_customer_date").isNotNull()
)

In [0]:
# Impute missing values for numerical column (payment_value)

from pyspark.ml.feature import Imputer
from pyspark.sql.functions import col, when, lit

# Step 1: Create nulls (for demonstration)
payments_df_cleaned = payments_df.withColumn(
    "payment_value",
    when(col("payment_value") != 99.33, col("payment_value")).otherwise(lit(None))
)

# Step 2: Apply imputer on cleaned dataframe
imputer = Imputer(
    inputCols=["payment_value"],
    outputCols=["payment_value_imputed"]
).setStrategy("mean")

payments_df_cleaned = imputer.fit(payments_df_cleaned).transform(payments_df_cleaned)

payments_df_cleaned.show()

In [0]:
# impute missing values
from pyspark.ml.feature import Imputer

impute = Imputer(inputCols=['payment_value'],outputCols=['payment_value_imputed']).setStrategy('mean')
# adding a null value to show how impute works 
from pyspark.sql.functions import lit, when, col

payment_df_cleaned=payment_df.withColumn("payment_value",when(col("payment_value")!=99.33,col('payment_value')).otherwise(lit(None)))

payment_df_cleaned = impute.fit(payment_df).transform(payment_df)
payment_df_cleaned.show()

### Standardizing the format


In [0]:
def print_schema(df,df_name):
  print(f"Schema of: {df_name}")
  df.printSchema()

In [0]:
print_schema(customers_df,"customers")

In [0]:
print_schema(orders_df,"orders")

In [0]:
print_schema(order_items_df,"order_items")

In [0]:
print_schema(reviews_df,"review")

In [0]:
print_schema(payment_df,"payments")

In [0]:
print_schema(sellers_df,"seller")

In [0]:
print_schema(geolocation_df,"geoloaction")

In [0]:
print_schema(products_df,"products")

- Payment Type Standardization

In [0]:
payment_df_cleaned = payment_df_cleaned.withColumn("payment_type", 
                                                   when(col("payment_type")=="boleto","Bank Transfer")\
                                                  .when(col("payment_type")=="credit_card","CC")\
                                                  .when(col("payment_type")=="debit_card","Debit card")\
                                                  .otherwise("other"))

In [0]:
payment_df_cleaned.show()

In [0]:
print_schema(customers_df,"customers")

In [0]:
# customer_zip_code is integer — converting to string
# to avoid joining/groupBy issues in downstream processing

customers_df_cleaned = customers_df.withColumn("customer_zip_code_prefix",col("customer_zip_code_prefix").cast("string"))

customers_df_cleaned.printSchema()

- Remove Duplicate Records

In [0]:
# Future task:- perform on all df and create a function for standardizing

customers_df_cleaned = customers_df_cleaned.dropDuplicates(['customer_id'])

## Silver Layer: Data Integration (Joins)

In [0]:
order_with_details= orders_df_cleaned\
 .join(order_items_df,"order_id","left")\
 .join(payment_df_cleaned,"order_id","left")\
 .join(customers_df_cleaned,"customer_id","left")

order_with_details.show() 


### Advance Transformations

In [0]:
# outlier detection - Compute quantiles
quantities_df = order_items_df.approxQuantile("price",[0.01, 0.99],0.00)
low_cutoff,high_cutoff = quantities_df[0],quantities_df[1]

low_cutoff,high_cutoff

In [0]:
# Filter outliers
order_items_df_cleaned = order_items_df.filter((col("price")>=low_cutoff) & (col("price")<= high_cutoff))
order_items_df_cleaned.show()

In [0]:
products_df_cleaned = products_df.withColumn("product_size_category",
     when(col("product_weight_g")< 500,"Small")\
    .when(col("product_weight_g").between(500,2000), "Medium")\
    .otherwise("Large"))

products_df_cleaned.show()